In [0]:
# ============================================================
# 02_vendor_agent_tools — Stage 2
#   Governed UC Function tools over gold_vendor_performance,
#   for the Vendor Performance sub-agent.
# ------------------------------------------------------------
# Tool 1: get_vendor_performance(vendor_id)
#   Deep-dive on ONE vendor. Includes a data_confidence flag
#   so the agent knows when a vendor's rates rest on few orders.
# ============================================================

spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.get_vendor_performance(
    p_vendor_id STRING COMMENT 'Vendor identifier, e.g. V008 or V014'
)
RETURNS TABLE (
    vendor_id            STRING,
    vendor_name          STRING,
    country              STRING,
    total_po_lines       BIGINT,
    otif_rate            DECIMAL(6,4),
    on_time_rate         DECIMAL(6,4),
    in_full_rate         DECIMAL(6,4),
    rejection_rate       DECIMAL(6,4),
    avg_cycle_time_days  DECIMAL(18,2),
    total_order_value    DECIMAL(22,2),
    currency             STRING,
    data_confidence      STRING
)
COMMENT 'Returns the full performance profile for one vendor: OTIF, on-time, in-full, rejection rates, cycle time and order value. Use for questions about a single specific vendor. The data_confidence field warns when a vendor has too few orders for its rates to be reliable.'
RETURN
    SELECT
        vendor_id, vendor_name, country, total_po_lines,
        otif_rate, on_time_rate, in_full_rate, rejection_rate,
        avg_cycle_time_days, total_order_value, currency,
        CASE WHEN total_po_lines < 30 THEN 'LOW - too few orders, treat rates with caution'
             ELSE 'OK' END AS data_confidence
    FROM workspace.default.gold_vendor_performance
    WHERE vendor_id = p_vendor_id
""")

print("get_vendor_performance created.")
print("Test - V008 (high-volume vendor):")
spark.sql("SELECT * FROM workspace.default.get_vendor_performance('V008')").show(truncate=False)
print("Test - V017 (1-order vendor, should flag LOW confidence):")
spark.sql("SELECT * FROM workspace.default.get_vendor_performance('V017')").show(truncate=False)

In [0]:
# ============================================================
# Cell 2 — Tool 2: rank_vendors_by_metric(metric, top_n, worst_first)
#   Ranks vendors by a chosen performance metric. Excludes
#   low-data vendors (< 30 PO lines) so artifacts like a
#   1-order 100% OTIF vendor cannot distort the ranking.
# ============================================================

spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.rank_vendors_by_metric(
    p_metric      STRING COMMENT 'Metric to rank by: otif, on_time, rejection, or cycle_time',
    p_top_n       INT    COMMENT 'How many vendors to return, e.g. 5',
    p_worst_first STRING COMMENT 'Set to yes to show worst performers first; no for best first'
)
RETURNS TABLE (
    rank_position        INT,
    vendor_id            STRING,
    vendor_name          STRING,
    total_po_lines       BIGINT,
    otif_rate            DECIMAL(6,4),
    on_time_rate         DECIMAL(6,4),
    rejection_rate       DECIMAL(6,4),
    avg_cycle_time_days  DECIMAL(18,2)
)
COMMENT 'Ranks vendors by a chosen metric (otif, on_time, rejection, or cycle_time) and returns the top N. Use for comparison or ranking questions such as which vendors perform best or worst. Set worst_first to yes for worst performers. Vendors with fewer than 30 purchase-order lines are excluded so unreliable low-data vendors do not distort the ranking.'
RETURN
    SELECT rank_position, vendor_id, vendor_name, total_po_lines,
           otif_rate, on_time_rate, rejection_rate, avg_cycle_time_days
    FROM (
        SELECT
            CAST(ROW_NUMBER() OVER (
                ORDER BY
                  CASE
                    WHEN p_metric = 'otif'       AND p_worst_first = 'yes' THEN otif_rate
                    WHEN p_metric = 'otif'                                 THEN -otif_rate
                    WHEN p_metric = 'on_time'    AND p_worst_first = 'yes' THEN on_time_rate
                    WHEN p_metric = 'on_time'                              THEN -on_time_rate
                    WHEN p_metric = 'rejection'  AND p_worst_first = 'yes' THEN -rejection_rate
                    WHEN p_metric = 'rejection'                            THEN rejection_rate
                    WHEN p_metric = 'cycle_time' AND p_worst_first = 'yes' THEN -avg_cycle_time_days
                    WHEN p_metric = 'cycle_time'                           THEN avg_cycle_time_days
                    ELSE -otif_rate
                  END
            ) AS INT)            AS rank_position,
            vendor_id, vendor_name, total_po_lines,
            otif_rate, on_time_rate, rejection_rate, avg_cycle_time_days
        FROM workspace.default.gold_vendor_performance
        WHERE total_po_lines >= 30
    )
    WHERE rank_position <= p_top_n
""")

print("rank_vendors_by_metric created.")
print("Test - top 5 vendors by OTIF (best first):")
spark.sql("SELECT * FROM workspace.default.rank_vendors_by_metric('otif', 5, 'no')").show(truncate=False)
print("Test - worst 5 vendors by OTIF:")
spark.sql("SELECT * FROM workspace.default.rank_vendors_by_metric('otif', 5, 'yes')").show(truncate=False)
print("Test - worst 3 by cycle time:")
spark.sql("SELECT * FROM workspace.default.rank_vendors_by_metric('cycle_time', 3, 'yes')").show(truncate=False)